# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Anish494/flyrank_ai_first_assignment/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding A — "What Predicts Health?" (Random Forest, ML Appendix, p.27)**
The model reports Average Position (43%) and Impressions (32%) as the top predictors
of health_score. My methodology question: since health_score is explicitly defined as
a weighted sum of impressions, position, CTR, and scroll depth, is this importance
ranking meaningfully different from what you'd get by just reading the health_score
formula itself? The paper honestly flags this ("importance is descriptive rather than
causal"), which I respect — my question is whether a version of this analysis built
only on features NOT used to construct health_score (e.g. word count, content age,
AI sessions) would reveal anything the formula doesn't already tell us, since that
version would carry more real predictive information.

**Finding B — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy,
p.28)**
The methodology page states an 80/20 split for this model, across a dataset spanning
57 brands. My methodology question: was this split grouped by brand, or was it a
plain random row split? I ask this because I found in my own Notebook 03 work that a
random split (letting a client's pages appear in both train and test) produced a
falsely optimistic result — accuracy that looked fine under a random split collapsed
below the base rate once I re-ran the same model with a GroupShuffleSplit on client
ID. If pages from the same brand can land in both the 71%-accuracy model's train and
test sets, the model may be partially learning brand-specific patterns rather than a
generalizable growth signal, and the true held-out accuracy across unseen brands could
be meaningfully lower than 71%.

In [1]:
import os, sys, subprocess
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import GroupShuffleSplit

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Anish494/flyrank_ai_first_assignment"
REPO_DIR = "flyrank_ai_first_assignment"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
cluster_features = ["impressions_90d", "avg_position", "ctr", "word_count", "content_age_days"]
print(f"{len(df):,} rows ready. Unique clients: {df['client_id'].nunique()}")

30,000 rows ready. Unique clients: 32


## 2. My model under an honest split (before/after)

**Before (ML-08's original check):** I split rows randomly into two halves
(train_test_split, no grouping), which allowed pages from the same client to appear
in both halves. This is the same gap I flagged in the paper's Finding B.

**After (this notebook):** I re-run the same stability check using GroupShuffleSplit
on client_id, so every client's pages land entirely in one half or the other — never
split across both.

In [2]:
# BEFORE: random row split (same as ML-08, reproduced here for direct comparison)
from sklearn.model_selection import train_test_split
before_a, before_b = train_test_split(df, test_size=0.5, random_state=1)
scaler_before = StandardScaler()
Xb_a = scaler_before.fit_transform(before_a[cluster_features].fillna(0))
Xb_b = scaler_before.transform(before_b[cluster_features].fillna(0))
sil_before_a = silhouette_score(Xb_a, KMeans(n_clusters=5, random_state=42, n_init=10).fit_predict(Xb_a))
sil_before_b = silhouette_score(Xb_b, KMeans(n_clusters=5, random_state=42, n_init=10).fit_predict(Xb_b))
print(f"BEFORE (random row split) -- silhouette A: {sil_before_a:.3f}, B: {sil_before_b:.3f}")

# AFTER: grouped by client_id
gss = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=1)
idx_a, idx_b = next(gss.split(df, groups=df["client_id"]))
after_a, after_b = df.iloc[idx_a], df.iloc[idx_b]

overlap = set(after_a["client_id"]) & set(after_b["client_id"])
print(f"\nAFTER (grouped by client_id) -- clients overlapping between halves: {len(overlap)}")

scaler_after = StandardScaler()
Xa_a = scaler_after.fit_transform(after_a[cluster_features].fillna(0))
Xa_b = scaler_after.transform(after_b[cluster_features].fillna(0))
sil_after_a = silhouette_score(Xa_a, KMeans(n_clusters=5, random_state=42, n_init=10).fit_predict(Xa_a))
sil_after_b = silhouette_score(Xa_b, KMeans(n_clusters=5, random_state=42, n_init=10).fit_predict(Xa_b))
print(f"AFTER (grouped by client_id) -- silhouette A: {sil_after_a:.3f}, B: {sil_after_b:.3f}")

BEFORE (random row split) -- silhouette A: 0.385, B: 0.380

AFTER (grouped by client_id) -- clients overlapping between halves: 0
AFTER (grouped by client_id) -- silhouette A: 0.343, B: 0.377


**Finding:** the random split (BEFORE) showed silhouette 0.385 vs 0.380 — a gap of
just 0.005, looking very stable. The grouped split (AFTER, zero client overlap
confirmed) showed 0.343 vs 0.377 — a gap of 0.034, nearly 7x larger. This confirms
the same effect I found in Notebook 03 and flagged in the paper's Finding B: a random
split can make results look more stable/generalizable than they really are, because
pages from the same client leak information across the split. The honest answer is
that my clustering structure is still reasonably stable across truly unseen clients
(0.343 and 0.377 are both meaningfully above 0, and both close to the original full-
data silhouette of 0.384 from ML-08), but the random-split version overstated how
clean that stability was.

## 3. Leakage audit

Running the lane guide's leakage checklist against my five clustering features
(impressions_90d, avg_position, ctr, word_count, content_age_days):

- **Are any features calculated after the decision point?** No — all five are
  observed, present-window measurements from the starter dataset's 90-day window,
  not future-window values.
- **Does the feature window overlap the target window?** Not directly applicable in
  the same way as a supervised label — clustering has no future target window. But I
  already demonstrated in ML-04 that adding a feature derived from a *later* month
  (pct_change_mar_to_apr) created an artificial, meaningless split — proof that this
  risk is real for this lane even without a formal label.
- **If I rebuilt any product output, did it slip in as a normal feature?** No —
  none of my five features are FlyRank product decision flags (health_score,
  priority_score, action_type). All five are raw observed signals or simple derived
  measurements (content_age_days), per the lane guide's field-type categories.
- **Does a derived field secretly encode the target?** Clustering has no single
  target to encode, but I checked pairwise correlations between my five features
  back in ML-03 (all between -0.12 and 0.16) — none are redundant restatements of
  each other, which would have been a milder form of the same problem.
- **Are duplicate or related rows split across train and test in a way that makes
  the test too easy?** Yes, this WAS happening in ML-08's original stability check
  (random row split let the same client appear on both sides) — fixed above in
  Section 2 using GroupShuffleSplit, with zero client overlap confirmed.
- **Am I testing on clients or time periods the model hasn't effectively already
  seen?** After the fix in Section 2, yes — the grouped split guarantees genuinely
  unseen clients in each half.

**Conclusion:** the five features themselves are clean and observed-only. The real
leakage risk in my pipeline was methodological, not feature-based — the random split
in ML-08, now corrected.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

**Original claim (ML-08, Section 3):** "the baseline may be systematically
under-flagging exactly the weakest archetype, simply because it's also the
lowest-volume one."

**Rewritten, more careful version:** "In this dataset, pages in the weakest-performing
archetype (poor position, low CTR) were flagged by the baseline rule less often than
pages in stronger archetypes -- an observed pattern in this specific clustering run.
Since this clustering was validated for stability using a random split rather than a
grouped one (a gap this notebook corrects going forward), this specific archetype
boundary should be treated as directional rather than fully confirmed; the general
shape of the finding held up under a grouped re-check in Section 2 of this notebook,
but exact cluster membership for individual borderline pages may shift slightly
between random and grouped runs."

This uses safer language (observed, directional) instead of the more confident
"systematically" framing from the original, and explicitly ties the caveat to the
validation-design gap this notebook exists to catch.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.